In [52]:
# Imports and getting a handle on our data directories to start

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)


def sanitize_feature_name(name: str) -> str:
    """Normalize column names for stable deployment-safe feature names."""
    clean = re.sub(r'[^0-9a-zA-Z]+', '_', str(name).strip().lower()).strip('_')
    return clean or 'blank'


def prefixed_column_map(columns, prefix: str):
    """Create unique prefixed snake_case names, e.g. mech_alliances."""
    rename_map = {}
    used = set()
    for col in columns:
        base = f"{prefix}_{sanitize_feature_name(col)}"
        new_col = base
        i = 2
        while new_col in used:
            new_col = f"{base}_{i}"
            i += 1
        rename_map[col] = new_col
        used.add(new_col)
    return rename_map


def find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / '.git').exists() and (cand / 'data' / 'raw').exists():
            return cand
    return start

ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_DIR exists:', RAW_DIR.exists())
print('PROCESSED_DIR:', PROCESSED_DIR)


ROOT: C:\Users\guy74\Documents\NU Stuff\ANA680\Week 4\Final Project Board Game Ratings Predictions\Board Game Ratings Predictions
RAW_DIR exists: True
PROCESSED_DIR: C:\Users\guy74\Documents\NU Stuff\ANA680\Week 4\Final Project Board Game Ratings Predictions\Board Game Ratings Predictions\data\processed


In [53]:
# List repo tree (top levels) and raw files
for p in sorted(ROOT.iterdir()):
    if p.name.startswith('.git'):
        continue
    print(f"- {p.name}{'/' if p.is_dir() else ''}")

print('\nRaw CSV files:')
raw_csvs = sorted(RAW_DIR.glob('*.csv'))
for f in raw_csvs:
    print(f"- {f.name}")


- artifacts/
- data/
- notebooks/

Raw CSV files:
- artists_reduced.csv
- designers_reduced.csv
- games.csv
- mechanics.csv
- publishers_reduced.csv
- ratings_distribution.csv
- subcategories.csv
- themes.csv
- user_ratings.csv


In [54]:
# Get handles on the files we're going to use for the purposes of this project 
# (so games, mechanics, and subcategories only for now)
def detect_sources(raw_dir: Path):
    candidates = sorted(raw_dir.glob('*.csv'))
    if not candidates:
        raise FileNotFoundError(f'No CSV files found under {raw_dir}')

    mapping = {'games': None, 'mechanics': None, 'subcategories': None}

    # Name-based preference first
    for f in candidates:
        lname = f.name.lower()
        if mapping['games'] is None and lname == 'games.csv':
            mapping['games'] = f
        if mapping['mechanics'] is None and 'mechanic' in lname:
            mapping['mechanics'] = f
        if mapping['subcategories'] is None and ('subcategory' in lname or 'categories' in lname):
            mapping['subcategories'] = f

    return mapping

sources = detect_sources(RAW_DIR)
print('Detected sources:')
for k, v in sources.items():
    print(f'- {k}: {v.name if v else None}')



Detected sources:
- games: games.csv
- mechanics: mechanics.csv
- subcategories: subcategories.csv


In [55]:
# Define a function we'll call in the below cell to generate a simple report for each table we'll load in
def table_report(df: pd.DataFrame, table_name: str, key: str = 'BGGId'):
    rows, cols = df.shape

    print('\n' + '=' * 92)
    print(f'TABLE: {table_name}')
    print('-' * 92)
    print(f'Rows: {rows:,}')
    print(f'Columns: {cols:,}')

    if key not in df.columns:
        print(f"WARNING: primary key '{key}' is missing.")
        key_nunique = None
        key_dupes = None
    else:
        total = len(df)
        key_nunique = df[key].nunique(dropna=False)
        key_nulls = int(df[key].isna().sum())
        key_dupes = total - key_nunique

        print(f"Primary key: {key}")
        print(f"  - total rows: {total:,}")
        print(f"  - unique keys: {key_nunique:,}")
        print(f"  - duplicate keys: {key_dupes:,}")
        print(f"  - null keys: {key_nulls:,}")

        if key_dupes > 0:
            print('  WARNING: Duplicate keys found. Top 10 duplicate BGGId values:')
            dup_counts = df[key].value_counts(dropna=False)
            print(dup_counts[dup_counts > 1].head(10).to_dict())
        else:
            print('  OK: Primary key is unique.')

    missing_count = df.isna().sum()
    missing_pct = (missing_count / rows * 100).round(2)

    missing_report = pd.DataFrame({
        'column': missing_count.index,
        'missing_count': missing_count.values,
        'missing_pct': missing_pct.values,
        'non_missing_count': rows - missing_count.values
    }).sort_values(['missing_count', 'column'], ascending=[False, True]).reset_index(drop=True)

    print('\nColumn-level missingness (all columns):')
    display(missing_report)

    return {
        'table': table_name,
        'rows': rows,
        'columns': cols,
        'key_present': key in df.columns,
        'key_unique': key_nunique,
        'key_duplicates': key_dupes,
    }


In [56]:
# Load source tables
games = pd.read_csv(sources['games'])
mechanics = pd.read_csv(sources['mechanics'])
subcats = pd.read_csv(sources['subcategories'])

# Normalize only colon-delimited game columns (e.g., Rank:boardgame -> Rank_boardgame)
games_colon_rename = {c: c.replace(':', '_') for c in games.columns if ':' in c}
if games_colon_rename:
    renamed_values = list(games_colon_rename.values())
    if len(renamed_values) != len(set(renamed_values)):
        raise ValueError('Colon replacement created duplicate games column names. Resolve before continuing.')
    games = games.rename(columns=games_colon_rename)
    print(f'Renamed {len(games_colon_rename)} games columns by replacing ":" with "_".')

reports = []
reports.append(table_report(games, 'games'))
reports.append(table_report(mechanics, 'mechanics'))
reports.append(table_report(subcats, 'subcategories'))

print('\n' + '=' * 92)
print('EXECUTIVE SUMMARY (TABLE SIZE + PRIMARY KEY CHECKS)')
display(pd.DataFrame(reports))


Renamed 17 games columns by replacing ":" with "_".

TABLE: games
--------------------------------------------------------------------------------------------
Rows: 21,925
Columns: 48
Primary key: BGGId
  - total rows: 21,925
  - unique keys: 21,925
  - duplicate keys: 0
  - null keys: 0
  OK: Primary key is unique.

Column-level missingness (all columns):


,column,missing_count,missing_pct,non_missing_count
0,Family,15262,69.61,6663
1,LanguageEase,5891,26.87,16034
2,ComAgeRec,5530,25.22,16395
3,ImagePath,17,0.08,21908
4,Description,1,0.00,21924
5,AvgRating,0,0.00,21925
6,BGGId,0,0.00,21925
7,BayesAvgRating,0,0.00,21925
8,BestPlayers,0,0.00,21925
9,Cat_Abstract,0,0.00,21925



TABLE: mechanics
--------------------------------------------------------------------------------------------
Rows: 21,925
Columns: 158
Primary key: BGGId
  - total rows: 21,925
  - unique keys: 21,925
  - duplicate keys: 0
  - null keys: 0
  OK: Primary key is unique.

Column-level missingness (all columns):


,column,missing_count,missing_pct,non_missing_count
0,Action Drafting,0,0.0,21925
1,Action Points,0,0.0,21925
2,Action Queue,0,0.0,21925
3,Action Retrieval,0,0.0,21925
4,Action Timer,0,0.0,21925
...,...,...,...,...
153,Variable Set-up,0,0.0,21925
154,Victory Points as a Resource,0,0.0,21925
155,Voting,0,0.0,21925
156,Worker Placement,0,0.0,21925



TABLE: subcategories
--------------------------------------------------------------------------------------------
Rows: 21,925
Columns: 11
Primary key: BGGId
  - total rows: 21,925
  - unique keys: 21,925
  - duplicate keys: 0
  - null keys: 0
  OK: Primary key is unique.

Column-level missingness (all columns):


,column,missing_count,missing_pct,non_missing_count
0,BGGId,0,0.0,21925
1,Card Game,0,0.0,21925
2,Collectible Components,0,0.0,21925
3,Educational,0,0.0,21925
4,Electronic,0,0.0,21925
5,Exploration,0,0.0,21925
6,Miniatures,0,0.0,21925
7,Print & Play,0,0.0,21925
8,Puzzle,0,0.0,21925
9,Territory Building,0,0.0,21925



EXECUTIVE SUMMARY (TABLE SIZE + PRIMARY KEY CHECKS)


,table,rows,columns,key_present,key_unique,key_duplicates
0,games,21925,48,True,21925,0
1,mechanics,21925,158,True,21925,0
2,subcategories,21925,11,True,21925,0


In [57]:
# Some checks prior to the join to ensure no dupes and no key missing fields like target/primaryID, etc
required_cols = {'BGGId', 'AvgRating'}
missing_required = required_cols - set(games.columns)
if missing_required:
    raise KeyError(f'games table missing required columns: {sorted(missing_required)}')

# Enforce one-row-per-game at source (fail fast instead of silently deduping)
duplicate_keys = int(games['BGGId'].duplicated().sum())
if duplicate_keys > 0:
    raise ValueError(f'games has {duplicate_keys} duplicate BGGId rows; resolve upstream before joins.')

# Ensure target is strictly numeric and fully populated
games['AvgRating'] = pd.to_numeric(games['AvgRating'], errors='raise')
missing_target = int(games['AvgRating'].isna().sum())
if missing_target > 0:
    raise ValueError(f'games has {missing_target} missing AvgRating values.')

base_n = len(games)
print(f'games integrity checks passed | base row count: {base_n:,}')


games integrity checks passed | base row count: 21,925


In [58]:
# Join in mechanics and do basic checks/reporting after the join
if 'BGGId' not in mechanics.columns:
    raise KeyError('mechanics table is missing BGGId')

mech = mechanics.copy()
raw_mech_cols = [c for c in mech.columns if c != 'BGGId']

# Prefix mechanics feature columns and normalize names to snake_case
mech_rename_map = prefixed_column_map(raw_mech_cols, 'mech')

# Coerce one-hot columns to numeric in one vectorized pass (avoids fragmentation warning)
mech_numeric = mech[raw_mech_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
mech_numeric = mech_numeric.rename(columns=mech_rename_map)
mech_cols = list(mech_numeric.columns)

mech = pd.concat([mech[['BGGId']], mech_numeric], axis=1)

pre_dup_count = int(mech['BGGId'].duplicated().sum())
print('Mechanics duplicate-key rows before aggregation:', pre_dup_count)

if pre_dup_count > 0:
    mech_agg = mech.groupby('BGGId', as_index=False)[mech_cols].max()
else:
    # Already one row per BGGId, no aggregation needed
    mech_agg = mech

print('Mechanics shape after aggregation:', mech_agg.shape)

master = games.merge(mech_agg, on='BGGId', how='left', validate='one_to_one')
print('Row count after mechanics join:', len(master))
assert len(master) == base_n, 'Row count changed after mechanics join'

# Indicator for mechanics availability
master['HasMechanicsData'] = (master[mech_cols].fillna(0).sum(axis=1) > 0).astype(int)
total_games = len(master)
games_with_mech = int(master['HasMechanicsData'].sum())
games_without_mech = total_games - games_with_mech
print('\nMechanics coverage summary:')
print('Total games:', total_games)
print('Games with >=1 mechanic:', games_with_mech)
print('Games with no mechanics flagged:', games_without_mech)
print('Pct with mechanics:', round(games_with_mech / total_games * 100, 2), '%')


Mechanics duplicate-key rows before aggregation: 0
Mechanics shape after aggregation: (21925, 158)
Row count after mechanics join: 21925

Mechanics coverage summary:
Total games: 21925
Games with >=1 mechanic: 20841
Games with no mechanics flagged: 1084
Pct with mechanics: 95.06 %


In [59]:
# Join in subcategories and do basic checks/reporting after the join
subcat_cols_final = []
if subcats is not None:
    if 'BGGId' not in subcats.columns:
        print('Subcategories file has no BGGId; skipping subcategories join.')
    else:
        sub = subcats.copy()
        non_key = [c for c in sub.columns if c != 'BGGId']

        # Heuristic: wide one-hot if most non-key columns are numeric/binary-like
        sample_non_key = sub[non_key].head(200) if non_key else pd.DataFrame()
        numeric_like_ratio = 0.0
        if len(non_key) > 0:
            num_like = 0
            for c in non_key:
                conv = pd.to_numeric(sample_non_key[c], errors='coerce')
                if conv.notna().mean() > 0.9:
                    num_like += 1
            numeric_like_ratio = num_like / len(non_key)

        is_probably_wide = numeric_like_ratio >= 0.7
        print('Subcategories detected as wide one-hot:', is_probably_wide)

        if is_probably_wide:
            # Prefix subcategory feature columns and normalize names to snake_case
            sub_rename_map = prefixed_column_map(non_key, 'subcat')
            sub_numeric = sub[non_key].apply(pd.to_numeric, errors='coerce').fillna(0)
            sub_numeric = sub_numeric.rename(columns=sub_rename_map)
            subcat_cols_final = list(sub_numeric.columns)

            sub = pd.concat([sub[['BGGId']], sub_numeric], axis=1)
            sub_dup_count = int(sub['BGGId'].duplicated().sum())
            if sub_dup_count > 0:
                sub_agg = sub.groupby('BGGId', as_index=False)[subcat_cols_final].max()
            else:
                sub_agg = sub
        else:
            # Long format fallback: infer category label column and pivot to one-hot
            cat_col_candidates = [c for c in non_key if any(x in c.lower() for x in ['subcat', 'category', 'name', 'label', 'value'])]
            cat_col = cat_col_candidates[0] if cat_col_candidates else (non_key[0] if non_key else None)
            if cat_col is None:
                print('Could not infer subcategory label column; skipping subcategories join.')
                sub_agg = None
            else:
                long_df = sub[['BGGId', cat_col]].dropna().copy()
                long_df[cat_col] = long_df[cat_col].astype(str).str.strip()
                long_df = long_df[long_df[cat_col] != '']
                pivot = pd.crosstab(long_df['BGGId'], long_df[cat_col]).reset_index()

                raw_pivot_cols = [c for c in pivot.columns if c != 'BGGId']
                sub_rename_map = prefixed_column_map(raw_pivot_cols, 'subcat')
                sub_agg = pivot.rename(columns=sub_rename_map)
                subcat_cols_final = [sub_rename_map[c] for c in raw_pivot_cols]

        if 'sub_agg' in locals() and sub_agg is not None:
            before_join = len(master)
            master = master.merge(sub_agg, on='BGGId', how='left', validate='one_to_one')
            print('Row count after subcategories join:', len(master))
            assert len(master) == before_join == base_n, 'Row count changed after subcategories join'
            if subcat_cols_final:
                master['HasSubcategoryData'] = (master[subcat_cols_final].fillna(0).sum(axis=1) > 0).astype(int)
                print('Games with >=1 subcategory flag:', int(master['HasSubcategoryData'].sum()))
                print('Games with no subcategory flag:', int((master['HasSubcategoryData'] == 0).sum()))
        else:
            subcat_cols_final = []
else:
    print('No subcategories file provided; skipping this join.')


Subcategories detected as wide one-hot: True
Row count after subcategories join: 21925
Games with >=1 subcategory flag: 10033
Games with no subcategory flag: 11892


In [60]:
# Final validation
assert master['BGGId'].nunique(dropna=False) == len(master), 'Master is not one-row-per-BGGId'
assert 'AvgRating' in master.columns, 'Target AvgRating missing from master'

# Safe fallback when running this cell independently/out-of-order
subcat_cols_final = globals().get('subcat_cols_final', [])

mechanics_in_master = [c for c in mech_cols if c in master.columns]
subcats_in_master = [c for c in subcat_cols_final if c in master.columns] if subcat_cols_final else []

print('Final master shape:', master.shape)
print('Unique BGGId count:', master['BGGId'].nunique())
print('Mechanics feature count:', len(mechanics_in_master))
print('Subcategory feature count:', len(subcats_in_master))

missing_summary = master.isna().mean().sort_values(ascending=False).head(25)
print('\nTop missingness (fraction):')
print(missing_summary)


Final master shape: (21925, 217)
Unique BGGId count: 21925
Mechanics feature count: 157
Subcategory feature count: 10

Top missingness (fraction):
Family            0.696100
LanguageEase      0.268689
ComAgeRec         0.252223
ImagePath         0.000775
Description       0.000046
AvgRating         0.000000
BayesAvgRating    0.000000
BGGId             0.000000
Name              0.000000
MinPlayers        0.000000
StdDev            0.000000
MaxPlayers        0.000000
BestPlayers       0.000000
GoodPlayers       0.000000
NumOwned          0.000000
NumWant           0.000000
NumWish           0.000000
NumWeightVotes    0.000000
MfgPlaytime       0.000000
ComMinPlaytime    0.000000
ComMaxPlaytime    0.000000
MfgAgeRec         0.000000
NumUserRatings    0.000000
YearPublished     0.000000
GameWeight        0.000000
dtype: float64


In [61]:
out_csv = PROCESSED_DIR / 'bgg_master.csv'
master.to_csv(out_csv, index=False)
print('Saved:', out_csv)

# Safe fallback when running this cell independently/out-of-order
subcat_cols_final = globals().get('subcat_cols_final', [])
mechanics_in_master = globals().get('mechanics_in_master', [c for c in mech_cols if c in master.columns])
subcats_in_master = [c for c in subcat_cols_final if c in master.columns] if subcat_cols_final else []

dd_path = PROCESSED_DIR / 'data_dictionary.md'
with open(dd_path, 'w', encoding='utf-8') as f:
    f.write('# BGG Master Dataset Data Dictionary\n\n')
    f.write('## Sources and Join Strategy\n')
    f.write('- Base table: games (1 row per BGGId)\n')
    f.write('- Column normalization: replaced ":" with "_" in games column names (e.g., Rank_boardgame)\n')
    f.write('- Left join: mechanics (deduplicated by BGGId using max across one-hot columns)\n')
    if subcat_cols_final:
        f.write('- Left join: subcategories (wide one-hot or long-to-wide pivot, then 1 row per BGGId)\n')
    else:
        f.write('- Subcategories: not included (missing/unusable source)\n')
    f.write('\n## Validation Rules\n')
    f.write('- Row count preserved after every join\n')
    f.write('- Final assert: nunique(BGGId) == total rows\n')
    f.write('- Target column present: AvgRating\n\n')
    f.write('## Column Summary\n')
    f.write(f'- Total columns: {master.shape[1]}\n')
    f.write(f'- Mechanics columns: {len(mechanics_in_master)}\n')
    f.write(f'- Subcategory columns: {len(subcats_in_master)}\n')

print('Saved:', dd_path)


Saved: C:\Users\guy74\Documents\NU Stuff\ANA680\Week 4\Final Project Board Game Ratings Predictions\Board Game Ratings Predictions\data\processed\bgg_master.csv
Saved: C:\Users\guy74\Documents\NU Stuff\ANA680\Week 4\Final Project Board Game Ratings Predictions\Board Game Ratings Predictions\data\processed\data_dictionary.md
